# Distributed BERT benchmark

This notebook renders and submits the same shared workload used by the command-line Skiff and native Slurm adapters. Review the resolved job before running the final submission cell.

In [ ]:
from datetime import datetime, timezone
import hashlib
import os
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from notebook_helpers import build_job, submit_job

GPU = False
SETTINGS = []  # Examples: ['samples=16000', 'epochs=2', 'nodes=5']
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ').lower()
NAME = f'bert-{RUN_ID}'
NAMESPACE = 'bert-benchmark'
STORAGE = '/root/.skiff/benchmarks/bert-distr-training'
CPU_IMAGE = '/root/.skiff/images/bert-training-cpu.sif'
GPU_IMAGE = '/root/.skiff/images/bert-training-gpu.sif'
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


In [ ]:
image = GPU_IMAGE if GPU else CPU_IMAGE
image_sha256 = sha256(image)
job = build_job(
    name=NAME, namespace=NAMESPACE, image=image, image_sha256=image_sha256,
    run_dir=f'results/{RUN_ID}', storage=STORAGE, gpu=GPU, settings=SETTINGS,
)
replicas = job['spec']['pytorchReplicaSpecs']
print('Job:', f'{NAMESPACE}/{NAME}')
print('Nodes:', sum(spec['replicas'] for spec in replicas.values()))
print('Resources:', replicas['Master']['template']['spec']['containers'][0]['resources'])
job


## Submit

The next cell creates the PyTorchJob. It does not impose a workload timeout.

In [ ]:
submitted = submit_job(job)
print('Submitted:', submitted['metadata']['name'])
